In [1]:
# make sure jupyter server is installed in the environment
# then install dependencies
%pip install pandas nltk scikit-learn numpy matplotlib symspellpy setuptools --quiet

from config import get_merged_dataframe
from main import configure

configure()

df = get_merged_dataframe(
    './data/processedNegative.csv',
    './data/processedPositive.csv',
    './data/processedNeutral.csv',
)

df.sample(10).reset_index(drop=True)

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt_tab to /home/samy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/samy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/samy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,tweet,sentiment
0,You say VUCA,positive
1,It was a new mother who challenged change in d...,neutral
2,this wasnt meant to sound ano sad,negative
3,a menu for judges,neutral
4,instant message perfectly fine :D how about yo...,positive
5,where are you now? happy,positive
6,Only to ways to tackle world's says Prabhat P...,neutral
7,that might have cost you $3 a month!!!! happy,positive
8,I know I'm not ready to say goodbye unhappy So,negative
9,Justice faces Supreme Court,neutral


In [2]:
from tokenizer import (
    lemmatize_tokens,
    stem_tokens,
    snowball_stem_tokens,
    lancaster_stem_tokens,
    misspell_and_lemmatize_tokens,
    misspell_tokens
)
from vectorizer import tfidf_vectorize, count_vectorize, binary_vectorize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB, ComplementNB

config = {
    "tokenization": {
        "Tokenization": None,
        "Lemmatization": lemmatize_tokens,
        "Stemming": stem_tokens,
        "Stemming Snowball": snowball_stem_tokens,
        "Stemming Lancaster": lancaster_stem_tokens,
        "Misspellings": misspell_tokens,
        "Misspellings + Lemmatization": misspell_and_lemmatize_tokens,
    },
    "vectorization": {
        "TF-IDF": tfidf_vectorize,
        "Count": count_vectorize,
        "Binary Count": binary_vectorize,
    },
    "classifiers": [
        LogisticRegression(),
        RandomForestClassifier(),
        MultinomialNB(),
        SVC(),
        BernoulliNB(),
        ComplementNB(),
    ],
}

/home/samy/tweets/tokenizer.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Classification

In [3]:
from train import train_model, evaluate_model
import warnings
warnings.filterwarnings('ignore', message='The parameter.*token_pattern.*will not be used')

results = {}

import pandas as pd

# Grid Search over all combinations
for model in config['classifiers']:
    # For each vectorization technique
    results[model.__class__.__name__] = pd.DataFrame(columns=config['vectorization'].keys(), index=config['tokenization'].keys())
    for vectorizer_name, vectorizer_func in config['vectorization'].items():
        # For each tokenization technique
        for tokenizer_name, tokenizer_func in config['tokenization'].items():
            # Print current combination
            print(f"Training {model.__class__.__name__}, {vectorizer_name}, {tokenizer_name}... ")
            model, _, x_test, y_test = train_model(
                df,
                tweet_column='tweet',
                sentiment_column='sentiment',
                tokenizer=tokenizer_func,
                vectorizer=vectorizer_func,
                classifier=model,
            )
            acc = evaluate_model(model, x_test, y_test)
            results[model.__class__.__name__].at[tokenizer_name, vectorizer_name] = acc
            # multiply by 100 to get percentage
            print(f"Accuracy: {acc:.2%}")
            print("-" * 50)

Training LogisticRegression, TF-IDF, Tokenization... 
Accuracy: 87.72%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Lemmatization... 
Accuracy: 88.58%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming... 
Accuracy: 88.58%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming Snowball... 
Accuracy: 88.44%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming Lancaster... 
Accuracy: 88.73%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Misspellings... 
Accuracy: 87.43%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Misspellings + Lemmatization... 
Accuracy: 86.99%
--------------------------------------------------
Training LogisticRegression, Count, Tokenization... 
Accuracy: 88.73%
--------------------------------------------------
T

In [4]:
for model in config['classifiers']:
    print(f"Results for {model.__class__.__name__}:")
    display(results[model.__class__.__name__])

Results for LogisticRegression:


,TF-IDF,Count,Binary Count
Tokenization,0.877168,0.887283,0.893064
Lemmatization,0.885838,0.887283,0.890173
Stemming,0.885838,0.885838,0.894509
Stemming Snowball,0.884393,0.885838,0.893064
Stemming Lancaster,0.887283,0.888728,0.890173
Misspellings,0.874277,0.874277,0.874277
Misspellings + Lemmatization,0.869942,0.872832,0.868497


Results for RandomForestClassifier:


,TF-IDF,Count,Binary Count
Tokenization,0.890173,0.890173,0.888728
Lemmatization,0.888728,0.890173,0.880058
Stemming,0.888728,0.891618,0.895954
Stemming Snowball,0.887283,0.893064,0.882948
Stemming Lancaster,0.891618,0.882948,0.894509
Misspellings,0.872832,0.862717,0.865607
Misspellings + Lemmatization,0.871387,0.868497,0.859827


Results for MultinomialNB:


...

,TF-IDF,Count,Binary Count
Tokenization,0.884393,0.878613,0.887283
Lemmatization,0.878613,0.882948,0.885838
Stemming,0.884393,0.882948,0.885838
Stemming Snowball,0.884393,0.882948,0.885838
Stemming Lancaster,0.887283,0.884393,0.890173
Misspellings,0.875723,0.869942,0.875723
Misspellings + Lemmatization,0.874277,0.871387,0.872832


Results for BernoulliNB:


,TF-IDF,Count,Binary Count
Tokenization,0.885838,0.885838,0.885838
Lemmatization,0.885838,0.885838,0.885838
Stemming,0.893064,0.893064,0.893064
Stemming Snowball,0.893064,0.893064,0.893064
Stemming Lancaster,0.895954,0.895954,0.895954
Misspellings,0.871387,0.871387,0.871387
Misspellings + Lemmatization,0.877168,0.877168,0.877168


Results for ComplementNB:


,TF-IDF,Count,Binary Count
Tokenization,0.833815,0.852601,0.849711
Lemmatization,0.843931,0.856936,0.858382
Stemming,0.851156,0.867052,0.864162
Stemming Snowball,0.851156,0.865607,0.864162
Stemming Lancaster,0.859827,0.862717,0.864162
Misspellings,0.822254,0.842486,0.83815
Misspellings + Lemmatization,0.83237,0.848266,0.846821
